# Lesson 10 — Multi-agent

**You will:** build a research-then-write pipeline where one agent calls another as a tool.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/10-multi-agent/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

Many real tasks are better modelled as a *team*: a researcher gathers information, a writer composes the report, a reviewer checks the writing.

Multi-agent isn't a new framework feature — it's a tiny composition trick. A `Bear` produces an answer from a task. A `Tool` produces a result from arguments. **Therefore: a Bear is a Tool.** BareBear ships this as one method:

```python
research_tool = researcher.as_tool(name="research", description="Research a topic")
```

Forty lines of Python and you have multi-agent.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")

# Optional: pin a specific free-tier model. If unset, barebear uses its default.
# Free-tier model availability rotates — swap here if a lesson cell errors with
# "model not available". Any *:free model on https://openrouter.ai/models works.
# os.environ["BAREBEAR_MODEL"] = "meta-llama/llama-3.2-3b-instruct:free"

print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")
print("Model:", os.environ.get("BAREBEAR_MODEL", "(framework default)"))

## A two-agent pipeline: researcher → writer

In [ ]:
from barebear import Bear, Task, Tool, OpenRouterModel

# fake search — replace with a real one in production
def search_web(query: str) -> str:
    facts = {
        "a-level cs": "UK A-level CS reform under consultation 2025: increased focus on AI ethics and software engineering.",
        "ib": "IB Computer Science: now includes a 'Computational Solutions' option.",
    }
    for k, v in facts.items():
        if k in query.lower():
            return v
    return "No specific findings."

researcher = Bear(
    model=OpenRouterModel(),
    tools=[Tool(name="search_web", fn=search_web, description="Search the web; returns a brief summary")],
)

writer = Bear(
    model=OpenRouterModel(),
    tools=[
        researcher.as_tool(
            name="research",
            description="Research a topic. Returns a brief synthesis. Use this whenever you need facts.",
        ),
    ],
)

report = writer.run(
    Task(goal="Write a short paragraph on UK A-level CS syllabus reform."),
    trace=True,
)
print()
print(report.final_output)

## What just happened

- The **writer** doesn't know how to search. It only has one tool: `research`.
- When the writer calls `research`, the **researcher** runs a complete sub-task with its own tool (`search_web`).
- The researcher returns its final answer as a string. The writer treats it as a tool result and continues.

Each agent has clear responsibility. Each is testable in isolation. You can swap models between them: a cheap fast model for research, a smarter model for writing.

## Exercise

1. Add a third agent — a `reviewer` Bear — that takes the writer's output, critiques it, and returns a revised version. Hint: it's `Reflect` with extra steps.
2. Trace both bears with `trace=True`. How many total LLM calls did the pipeline take? Compare with a single-agent version.

## What's next

[Lesson 11 →](../11-evaluation/lesson.ipynb)